# Base Feature Engineering
**Goal:** Create raw market data.
**Key Operations:**
1.  **Data Preparation:** Load raw Parquet data, process splits, and interpolate missing values.
2.  **Indicators:** Calculate a wide array of technical indicators (SMA, EMA, ATR, BB, etc.).

In [1]:
# --- Base Feature Engineering ---
# Goal: Create raw market data.

# 1) WIPE & PURGE
%reset -f
import gc
import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2) IMPORTS
import os
import psutil
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display

# Standardize display
pd.set_option('display.max_columns', None)

# 3) CUSTOM LIBRARIES
from libs import preps, plots, params, feats
importlib.reload(preps)
importlib.reload(plots)
importlib.reload(params)
importlib.reload(feats)

print(f"🚀 Memory Cleaned. Environment Ready for {params.ticker}")


🚀 Memory Cleaned. Environment Ready for AAPL


In [2]:
print(f"[process_splits] Loading original Parquet: {params.alpaca_parquet}")

# 1. Load from Parquet (Much faster than CSV)
df_input = pd.read_parquet(params.alpaca_parquet)

# 1. Cleaning and preparing base data 
df = preps.process_splits(df=df_input, ticker=params.ticker)
df = preps.prepare_interpolate_data(df=df)

# 2. Index Cleanup: Ensure 1-minute frequency and naive UTC
df.index = pd.to_datetime(df.index).floor('min').tz_localize(None)

# Save cleaned base data
params.to_parquet_with_progress(df, params.base_parquet)

df

[process_splits] Loading original Parquet: dfs/AAPL_0_alpaca.parquet
Executing [detect_and_adjust_splits]...
Detected forward split on 2020-08-31 08:00:00 with factor 4 (ratio: 0.2514)
SUCCESS: Adjusted 1 split events for AAPL.
[executing prepare_interpolate_data]
Normalizing timezones...
Interpolating missing bars...
Filtering flat sessions...


Saving Parquet: 100%|██████████| 96/96 [00:01<00:00, 67.61it/s]


,open,high,low,close,volume,trade_count,vwap,ask,bid
2015-12-31 23:00:00,26.3575,26.3575,26.3575,26.3575,700.000000,1.0,105.430000,26.3970,26.3180
2015-12-31 23:01:00,26.3500,26.3500,26.3500,26.3500,2020.000000,5.0,105.400000,26.3895,26.3105
2015-12-31 23:02:00,26.3500,26.3500,26.3500,26.3500,1593.333333,4.0,105.400000,26.3895,26.3105
2015-12-31 23:03:00,26.3500,26.3500,26.3500,26.3500,1166.666667,3.0,105.400000,26.3895,26.3105
2015-12-31 23:04:00,26.3500,26.3500,26.3500,26.3500,740.000000,2.0,105.400000,26.3895,26.3105
...,...,...,...,...,...,...,...,...,...
2026-03-13 15:41:00,252.1700,252.1800,252.0400,252.1000,54024.000000,1008.0,252.132389,252.1630,252.0370
2026-03-13 15:42:00,252.0850,252.1800,252.0402,252.1000,89398.000000,1339.0,252.109872,252.1630,252.0370
2026-03-13 15:43:00,252.1000,252.1000,251.8900,251.9301,99873.000000,2114.0,251.998252,251.9931,251.8671
2026-03-13 15:44:00,251.9150,251.9300,251.6598,251.7199,79542.000000,2049.0,251.769574,251.7828,251.6570


In [3]:
# --- TECHNICAL INDICATORS ---
# Tuned specifically for 1-minute scalping dynamics (adding micro-windows)
df_inds_unsc = feats.standard_indicators(
    df,
    ema=[2, 3, 5, 7, 9, 13, 21, 50],         
    atr=[2, 3, 5, 7, 9, 14, 21],                
    rsi=[2, 3, 5, 7, 10, 14, 21],               
    roc=[2, 3, 5, 10, 21],                      
    sma=[5, 9, 21, 50, 100],
    bb={20: [2.0, 3.0], 50: [2.0]},   
    vol_spike=[3, 7, 14, 28],          
    obv_roll=[3, 7, 21],                
    ret_std=[5, 21, 63],              
    macd=[
        {"fast": 3, "slow": 10, "signal": 4},  
        {"fast": 6, "slow": 13, "signal": 5},
        {"fast": 12, "slow": 26, "signal": 9},
    ],
    stoch=[
        {"k": 5, "d": 3, "smooth": 3},         
        {"k": 14, "d": 3, "smooth": 3},
    ],
    cci=[7, 14, 20],                   
    mfi=[7, 14, 20],                  
    cmf=[7, 14, 20],                   
    donch=[10, 20, 55],               
    roll_vwap=[10, 20, 50],            
    linreg_slope=[5, 20, 50],          
    kc=[
        {"ema_window": 20, "atr_window": 20, "atr_mult": 1.5},
        {"ema_window": 20, "atr_window": 20, "atr_mult": 2.0},
    ],
    psar={"step": 0.02, "max_step": 0.2},
)

# Save unscaled indicators 
params.to_parquet_with_progress(df_inds_unsc, params.indunsc_parquet)

df_inds_unsc

Initializing:   0%|          | 0/83 [00:00<?, ?it/s]

Saving Parquet: 100%|██████████| 94/94 [00:23<00:00,  3.99it/s]


,open,high,low,close,volume,trade_count,vwap,ask,bid,ret,log_ret,body,upper_shad,lower_shad,range_pct,sma_5,sma_pct_5,sma_9,sma_pct_9,sma_21,sma_pct_21,sma_50,sma_pct_50,sma_100,sma_pct_100,ema_2,ema_3,ema_5,ema_7,ema_9,ema_13,ema_21,ema_50,rsi_2,rsi_3,rsi_5,rsi_7,rsi_10,rsi_14,rsi_21,roc_2,roc_3,roc_5,roc_10,roc_21,cci_7,cci_14,cci_20,macd_line_3_10_4,macd_signal_3_10_4,macd_diff_3_10_4,macd_line_6_13_5,macd_signal_6_13_5,macd_diff_6_13_5,macd_line_12_26_9,macd_signal_12_26_9,macd_diff_12_26_9,stoch_k_5_3_3,stoch_d_5_3_3,stoch_k_14_3_3,stoch_d_14_3_3,atr_2,atr_pct_2,plus_di_2,minus_di_2,adx_2,atr_3,atr_pct_3,plus_di_3,minus_di_3,adx_3,atr_5,atr_pct_5,plus_di_5,minus_di_5,adx_5,atr_7,atr_pct_7,plus_di_7,minus_di_7,adx_7,atr_9,atr_pct_9,plus_di_9,minus_di_9,adx_9,atr_14,atr_pct_14,plus_di_14,minus_di_14,adx_14,atr_21,atr_pct_21,plus_di_21,minus_di_21,adx_21,bb_lband_20_2p0,bb_hband_20_2p0,bb_w_20_2p0,bb_lband_20_3p0,bb_hband_20_3p0,bb_w_20_3p0,bb_lband_50_2p0,bb_hband_50_2p0,bb_w_50_2p0,kc_mid_20_20_1.5,kc_l_20_20_1.5,kc_h_20_20_1.5,kc_w_20_20_1.5,kc_mid_20_20_2.0,kc_l_20_20_2.0,kc_h_20_20_2.0,kc_w_20_20_2.0,obv,obv_roll_3,obv_roll_7,obv_roll_21,mfi_7,mfi_14,mfi_20,cmf_7,cmf_14,cmf_20,vol_spike_3,vol_spike_7,vol_spike_14,vol_spike_28,donch_h_10,donch_l_10,donch_w_10,donch_h_20,donch_l_20,donch_w_20,donch_h_55,donch_l_55,donch_w_55,roll_vwap_10,roll_vwap_20,roll_vwap_50,slope_close_5,slope_close_20,slope_close_50,ret_std_5,ret_std_21,ret_std_63,vwap_ohlc_close_session,vwap_dist_session,psar,psar_dist,dist_high_100,dist_low_100
2016-01-04 08:47:00,25.9500,25.9500,25.9500,25.9500,520.0,2.0,103.800000,25.9889,25.9111,0.000241,0.000241,0.0000,0.0000,0.0000,0.000000,25.93750,0.000482,25.933056,0.000653,25.926310,0.000914,25.938275,0.000452,26.147824,-0.007566,25.946924,25.944217,25.940253,25.937570,25.935579,25.932864,25.933197,25.987678,98.684436,96.817308,94.203762,90.125845,77.309195,59.208009,43.243449,0.000482,0.000723,0.000868,0.000868,0.001109,131.154684,224.437928,262.411348,0.009462,0.006510,0.002952,0.005934,0.003808,0.002126,-0.004585,-0.011539,0.006954,1.000000,0.888889,1.000000,0.971429,0.005939,0.000229,98.684436,1.315564,93.528736,0.005205,0.000201,96.817213,3.182669,85.632759,0.004085,0.000157,94.028706,5.753319,67.289719,0.003662,0.000141,87.291639,9.110761,46.159542,0.003908,0.000151,72.549084,14.279671,29.271916,0.006309,0.000243,38.254340,24.700467,34.286782,0.009983,0.000385,21.487845,29.599048,51.826216,25.909948,25.943802,0.001306,25.901485,25.952265,0.001959,25.759513,26.117037,0.013784,25.932658,25.918347,25.946969,0.001104,25.932658,25.913577,25.951739,0.001472,1.666540e+05,1.661440e+05,1.650111e+05,1.476618e+05,82.617623,97.218643,98.149864,0.000000,0.000000,0.000000,1.061224,0.383610,0.225611,0.275639,25.95,25.9250,0.000963,25.95,25.915000,0.001349,26.375000,25.8750,0.019268,25.928732,25.926038,25.939678,0.006250,0.001300,-0.001818,0.000151,0.000116,0.002263,25.920139,0.001152,25.924524,0.000982,0.016474,0.002890
2016-01-04 08:48:00,25.9375,25.9375,25.9375,25.9375,800.0,1.0,103.750000,25.9764,25.8986,-0.000482,-0.000482,0.0000,0.0000,0.0000,0.000000,25.94000,-0.000096,25.934167,0.000129,25.927381,0.000390,25.929800,0.000297,26.143699,-0.007887,25.940641,25.940859,25.939335,25.937553,25.935963,25.933526,25.933588,25.985710,31.784664,43.992282,53.333229,56.847801,55.820183,48.713022,39.222311,-0.000241,0.000000,0.000482,0.000386,0.000868,13.625304,65.554010,105.555556,0.005605,0.006148,-0.000543,0.004901,0.004173,0.000728,-0.003918,-0.010015,0.006097,0.777778,0.925926,0.835165,0.945055,0.009219,0.000355,31.784664,68.215336,64.979703,0.007637,0.000294,43.992261,56.007686,61.093650,0.005768,0.000222,53.273148,46.603355,55.167383,0.004924,0.000190,55.635866,42.071181,41.548609,0.004863,0.000187,51.828427,38.762152,27.622085,0.006751,0.000260,33.195038,34.659154,31.991850,0.010103,0.000390,20.221922,33.746611,50.551650,25.911392,25.944608,0.001281,25.903088,25.952912,0.001922,25.798057,26.061543,0.010161,25.9331

In [4]:
# ======================================================================================
# INDICATOR SUITE VERIFICATION
# ======================================================================================

# 1. Check for expected feature groups
feature_cols = [c for c in df_inds_unsc.columns if c not in ['open', 'high', 'low', 'close', 'volume', 'vwap', 'trade_count']]

print(f"📊 Indicator Audit: {params.ticker}")
print(f"─── Total Features Created: {len(feature_cols)}")
print(f"─── Total Rows:             {len(df_inds_unsc):,}")

# 2. Check for missing data (NaNs) after the warmup drop
nan_counts = df_inds_unsc[feature_cols].isna().sum()
total_nans = nan_counts.sum()

if total_nans == 0:
    print(f"✅ Clean Slate: 0 NaNs detected in all feature columns.")
else:
    print(f"⚠️ Warning: {total_nans} NaNs found!")
    display(nan_counts[nan_counts > 0])

# 3. Micro-Window Sensitivity Check
# Verifying that the fast 3-period features are actually moving (high variance)
micro_features = [f for f in feature_cols if '_3' in f or '_5' in f]
print(f"─── Micro-Windows Detected: {len(micro_features)} features")

# 4. Show a sample of the key scalping features
sample_cols = ['close', 'vwap_dist_session', 'rsi_3', 'vol_spike_3', 'slope_close_5']
existing_sample = [c for c in sample_cols if c in df_inds_unsc.columns]

print("\n🚀 Key Scalping Feature Preview:")
display(df_inds_unsc[existing_sample].tail(5))

📊 Indicator Audit: AAPL
─── Total Features Created: 144
─── Total Rows:             2,337,739
✅ Clean Slate: 0 NaNs detected in all feature columns.
─── Micro-Windows Detected: 46 features

🚀 Key Scalping Feature Preview:


,close,vwap_dist_session,rsi_3,vol_spike_3,slope_close_5
2026-03-13 15:41:00,252.1000,-0.008352,12.622393,0.915335,-0.13760
2026-03-13 15:42:00,252.1000,-0.008296,12.622393,1.300290,-0.08120
2026-03-13 15:43:00,251.9301,-0.008898,5.783080,1.231505,-0.05798
2026-03-13 15:44:00,251.7199,-0.009668,2.883546,0.887703,-0.10101
2026-03-13 15:45:00,251.6800,-0.009784,2.523322,0.713264,-0.12201
